# 03 — LP Optimization & Results

Goals:
- Define battery parameters
- Solve LP optimization for each day in the dataset
- Compare three scenarios: fixed tariff / dynamic no battery / dynamic + battery
- Visualize: price curve + charge/discharge schedule for a representative day
- Compute yearly savings in EUR

## 0. Imports & configuration

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# VS Code sets __vsc_ipynb_file__ to the notebook's absolute path.
# We use it to find src/ regardless of where the Jupyter kernel was started.
_nb_dir = Path(globals()['__vsc_ipynb_file__']).parent
_src    = (_nb_dir / '../src').resolve()
sys.path.insert(0, str(_src))

from battery_utils import optimize_day, threshold_strategy, backtest

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

DATA = '../../Data/'

# ── Battery parameters ────────────────────────────────────────────────────────
S_MAX  = 10.0   # total capacity (kWh)
P_MAX  =  5.0   # max charge/discharge power (kW)
ETA    =  0.95  # round-trip efficiency
S_INIT =  5.0   # starting state of charge — 50%

print(f"src path: {_src}")
print("Imports OK")

## 1. Load data

In [ ]:
# ── Prices ────────────────────────────────────────────────────────────────────
prices_raw = pd.read_csv(DATA + 'prices_be.csv')
prices_raw['timestamp'] = pd.to_datetime(prices_raw['timestamp'], utc=True)
prices_raw = prices_raw.set_index('timestamp').sort_index()

prices = prices_raw['price_eur_kwh'].resample('1h').mean().to_frame()
prices.index = prices.index.tz_convert('Europe/Brussels')
prices['price_eur_kwh'] = prices['price_eur_kwh'].interpolate(method='time')

# ── Load ──────────────────────────────────────────────────────────────────────
load_raw = pd.read_csv(DATA + 'load_profile.csv')
load_raw['timestamp'] = pd.to_datetime(load_raw['timestamp'])
load = load_raw.set_index('timestamp').sort_index()
load.columns = ['consumption']

# Align both datasets to the same date range
price_end  = prices.index.max().tz_localize(None)
load = load[load.index <= price_end]

print(f"Prices: {prices.index.min().date()} → {prices.index.max().date()}  ({len(prices):,} hrs)")
print(f"Load:   {load.index.min().date()} → {load.index.max().date()}  ({len(load):,} hrs)")

## 2. LP solver — single day

In [ ]:
# ── Single-day demo: 2022-01-03 (a typical winter weekday) ───────────────────
demo_date = '2022-01-03'

p_day = prices[prices.index.date == pd.Timestamp(demo_date).date()]['price_eur_kwh'].values
l_day = load[load.index.date == pd.Timestamp(demo_date).date()]['consumption'].values

res_lp   = optimize_day(p_day, l_day, S_MAX, P_MAX, ETA, S_INIT, cyclic=True)
res_rule = threshold_strategy(p_day, l_day, S_MAX, P_MAX, ETA, S_INIT)
cost_base = float(np.dot(p_day, l_day))

print(f"Baseline (no battery): {cost_base:.4f} EUR")
print(f"LP optimisation:       {res_lp['cost']:.4f} EUR  (saving {cost_base - res_lp['cost']:.4f})")
print(f"Threshold rule:        {res_rule['cost']:.4f} EUR  (saving {cost_base - res_rule['cost']:.4f})")

# Plot: price + charge/discharge + SOC for this day
hours = range(24)
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

ax = axes[0]
ax.bar(hours, p_day, color='steelblue', alpha=0.7)
ax.set_ylabel('Price (EUR/kWh)')
ax.set_title(f'LP optimisation — {demo_date}')

ax2 = axes[1]
ax2.bar(hours, res_lp['c'], color='green',  alpha=0.7, label='Charge')
ax2.bar(hours, [-x for x in res_lp['d']], color='red', alpha=0.7, label='Discharge')
ax2.axhline(0, color='black', lw=0.8)
ax2.set_ylabel('Power (kW)')
ax2.legend()

ax3 = axes[2]
ax3.plot(hours, res_lp['s'], color='orange', lw=2, marker='o', ms=4)
ax3.set_ylabel('State of charge (kWh)')
ax3.set_xlabel('Hour of day')
ax3.set_ylim(0, S_MAX + 1)
ax3.set_xticks(range(0, 24, 1))

plt.tight_layout()
plt.show()

## 3. Backtesting — full dataset 2022–2025

In [ ]:
# backtest() is defined in ../src/battery_utils.py and imported above

In [ ]:
import time
from pathlib import Path

# ── Run threshold strategy (fast — no LP) ─────────────────────────────────────
t0 = time.time()
bt_rule = backtest(prices, load, S_MAX, P_MAX, ETA, S_INIT, strategy='threshold')
print(f"Threshold done: {len(bt_rule)} days  ({time.time()-t0:.1f}s)")

# ── Run LP optimisation (slower — one LP per day) ─────────────────────────────
t0 = time.time()
bt_lp = backtest(prices, load, S_MAX, P_MAX, ETA, S_INIT, strategy='lp')
print(f"LP done:        {len(bt_lp)} days  ({time.time()-t0:.1f}s)")

# ── Save results to CSV ────────────────────────────────────────────────────────
out_dir = Path('../results')
out_dir.mkdir(exist_ok=True)

bt_lp.to_csv(out_dir / 'backtest_lp.csv')
bt_rule.to_csv(out_dir / 'backtest_threshold.csv')
print(f"Saved to {out_dir.resolve()}")

## 4. Compare scenarios

In [ ]:
# ── Summary table: annual costs & savings ─────────────────────────────────────
years = sorted(bt_lp['year'].unique())

rows = []
for y in years:
    base  = bt_lp[bt_lp['year'] == y]['cost_baseline'].sum()
    lp    = bt_lp[bt_lp['year'] == y]['cost'].sum()
    rule  = bt_rule[bt_rule['year'] == y]['cost'].sum()
    rows.append({
        'year':              y,
        'no battery (EUR)':  round(base, 2),
        'threshold (EUR)':   round(rule, 2),
        'LP (EUR)':          round(lp,   2),
        'saving threshold %': round((base - rule) / base * 100, 1),
        'saving LP %':        round((base - lp)   / base * 100, 1),
    })

summary = pd.DataFrame(rows).set_index('year')
print(summary.to_string())

## 5. Visualisation — representative day

In [ ]:
MONTH_NAMES = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']

fig, axes = plt.subplots(3, 1, figsize=(13, 14))

# ── Cumulative cost: all three scenarios ──────────────────────────────────────
ax = axes[0]
ax.plot(pd.to_datetime(bt_lp.index),   bt_lp['cost_baseline'].cumsum(),  color='gray',      lw=2, label='No battery')
ax.plot(pd.to_datetime(bt_rule.index), bt_rule['cost'].cumsum(),          color='coral',     lw=2, label='Threshold')
ax.plot(pd.to_datetime(bt_lp.index),   bt_lp['cost'].cumsum(),            color='steelblue', lw=2, label='LP')
ax.set_ylabel('Cumulative cost (EUR)')
ax.set_title('Cumulative cost 2022–2025')
ax.legend()

# ── Average daily cost by month ────────────────────────────────────────────────
ax2 = axes[1]
base_mo = bt_lp.groupby('month')['cost_baseline'].mean()
lp_mo   = bt_lp.groupby('month')['cost'].mean()
rule_mo = bt_rule.groupby('month')['cost'].mean()
x = np.arange(1, 13)
w = 0.25
ax2.bar(x - w,   base_mo.values, width=w, color='gray',      alpha=0.8, label='No battery')
ax2.bar(x,       rule_mo.values, width=w, color='coral',     alpha=0.8, label='Threshold')
ax2.bar(x + w,   lp_mo.values,   width=w, color='steelblue', alpha=0.8, label='LP')
ax2.set_xticks(x)
ax2.set_xticklabels(MONTH_NAMES)
ax2.set_ylabel('Avg daily cost (EUR)')
ax2.set_title('Average daily cost by month')
ax2.legend()

# ── Average daily saving by month (vs no battery) ─────────────────────────────
ax3 = axes[2]
ax3.bar(x - w/2, (base_mo - lp_mo).values,   width=w, color='steelblue', alpha=0.8, label='LP saving')
ax3.bar(x + w/2, (base_mo - rule_mo).values, width=w, color='coral',     alpha=0.8, label='Threshold saving')
ax3.plot(x, base_mo.values, color='gray', lw=1.5, linestyle='--', marker='o', ms=4, label='No battery (ref)')
ax3.set_xticks(x)
ax3.set_xticklabels(MONTH_NAMES)
ax3.set_ylabel('Avg daily saving vs no battery (EUR)')
ax3.set_title('Monthly saving vs no-battery baseline')
ax3.legend()

plt.suptitle('Backtesting results 2022–2025', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()